# CSA prompt–ROI mapping audit
Bước đầu của pilot đã duyệt. **Accelerator None / Internet ON**. Chỉ kiểm TRAIΝ10k + VALID5000, không dùng TEST, không chạy CLIP/UPT/caption model, không train. Thêm ba Dataset: full ROI134; UPT prompt all-actions-v2; objectdetectionecache. Không cần checkpoint, ảnh JPEG, bank130 hay API Secret. Sửa ba path ở cell đầu nếu mount khác. Giữ toàn bộ output và gửi ZIP review; cần kiểm mapping trước khi tạo cache CLIP prompt và chạy hai nhánh500update.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
PROMPTS=Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/csa_prompt_roi_audit_v1')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh runtime/output directory.')
for p in (FULL_ROOT,PROMPTS,DETECTIONS):
    if not p.exists():raise FileNotFoundError('Attach the three listed Datasets or correct their mount paths: '+str(p))
SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-audit-',dir='/tmp'))
LOG=SOURCE/'notebook.log'

REVISION='14c5bf0e6b19480d9666db4af517e930967c4ea1'
MAPPING={'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/run_prompt_mapping_audit.py': 'run_prompt_mapping_audit.py', 'experiments/csa_paper/test_prompt_adapter_data.py': 'test_prompt_adapter_data.py', 'experiments/csa_paper/test_prompt_adapter_model.py': 'test_prompt_adapter_model.py', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'make_prompt_adapter_notebook.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py'}
HASHES={'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/run_prompt_mapping_audit.py': '78a870571fa9930458c74603f9a283e4fd5b63dd55c052670c10145e5dad5063', 'experiments/csa_paper/test_prompt_adapter_data.py': 'a774dab306aec09aadea869953b1cc04e5592edf1e365b53f02e73258be92851', 'experiments/csa_paper/test_prompt_adapter_model.py': '152d7419238d9484c5b16e84fd987f3e9f68a57649f17019e5532860e144554a', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'b553a7d68efda2c679b2b0b470fa63f11d772782ce112d186d10b4c05ff55b34', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984'}

print('CPU-only prompt/ROI audit. No model download or training.',flush=True)
for original,destination in MAPPING.items():
    url='https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original
    payload=urllib.request.urlopen(url,timeout=90).read()
    if hashlib.sha256(payload).hexdigest()!=HASHES[original]:raise RuntimeError('Source SHA differs: '+original)
    target=SOURCE/destination;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(payload)
(SOURCE/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['PYTHONPATH']=str(SOURCE)
CONFIG=SOURCE/'config.json'
CONFIG.write_text(json.dumps(dict(full_root=str(FULL_ROOT),prompts=str(PROMPTS),detections=str(DETECTIONS),output=str(OUTPUT),wall_seconds=7200)),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CPU mapping active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('CPU mapping wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'run_prompt_mapping_audit.py','--config',CONFIG],timeout=7500)
    complete=json.loads((OUTPUT/'summary.json').read_text())['status']=='PROMPT_ROI_MAPPING_AUDIT_COMPLETE'
finally:
    OUTPUT.mkdir(parents=True,exist_ok=True)
    if LOG.exists():shutil.copy2(LOG,OUTPUT/'notebook.log')
    shutil.copy2(SOURCE/'source_receipt.json',OUTPUT/'source_receipt.json')
    sys.path.insert(0,str(SOURCE))
    from run_prompt_mapping_audit import archive
    archive(OUTPUT)
    print('Mapping audit complete; send review ZIP. This is not a trained caption model.' if complete else 'INCOMPLETE: send review ZIP/log; do not train from partial links.',flush=True)
